Exploring embeddings in Llama 3

Visualize and query token embeddings from a modern LLM




In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import login
login()

model_name = 'meta-llama/Llama-3.1-8B'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, torch_dtype = torch.float16)

# get the embedding matrix
embeddings = (model.embed_tokens.weight if hasattr(model, 'embed_tokens') else model.wte.weight) # shape -> [vocab_size, hidden_dim]
print(f'Embedding matrix shape: {embeddings.shape}')

def get_token_embedding(word):
  # get the embedding for a word (first token if multitoken)
  token_ids = tokenizer.encode(word)
  token_id = token_ids[0]
  token_text = tokenizer.decode([token_id])
  embedding = embeddings[token_id].float()
  return embedding, token_id, token_text

words = ["Python", "JavaScript", "Java", "snake", "coffee", "espresso", "tea"]
word_embeddings = {}

print("\n" + "=" * 50)
print("TOKEN EMBEDDINGS (Llama 3.1 8B)")
print("=" * 50)
for word in words:
    emb, tid, text = get_token_embedding(word)
    word_embeddings[word] = emb
    print(f"{word:12} -> token {tid:6} '{text}' -> [{emb[0]:.3f}, {emb[1]:.3f}, ...]")

def cosine_similarity(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

# compute cosine similarities
print("Cosine Similarity")
pairs = [("Python", "JavaScript"), ("coffee", "espresso"),
         ("Python", "snake"), ("Java", "coffee")]
for w1, w2 in pairs:
  sim = cosine_similarity(word_embeddings[w1],word_embeddings[w2])
  print(f"sim{w1:12}, {w2:12} = {sim:.4f}")

print("\nExpected: Python-JavaScript > Python-snake")
print("Java-coffee is interesting: programming language vs the drink!")

# ============================================================
# EMBEDDING MATRIX STATS
# ============================================================
print(f"\nVocabulary size:     {embeddings.shape[0]:,}")
print(f"Embedding dimension: {embeddings.shape[1]:,}")
print(f"Total parameters:    {embeddings.numel():,}")
print(f"Memory (FP16):       {embeddings.numel() * 2 / 1e6:.1f} MB")
print(f"Memory (FP32):       {embeddings.numel() * 4 / 1e6:.1f} MB")


